# 환경 설정

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))

## 데이터 모델 정의
- Pydantic
- TypeDict
- dataclass

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """ 영화에 대한 자세한 정보를 가진다. """
    title : str = Field(description="영화 제목")
    year: int = Field(description="영화의 개봉 연도")
    director : str = Field(description="영화 감독")
    rating : float = Field(description="영화의 평점. 10점 만점")

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model = "gpt-5-nano",
    temperature = 0
)

# LLM에 Data Model 적용
llm_with_structured = llm.with_structured_output(Movie)

In [ ]:
response = llm_with_structured.invoke("영화 타이타닉에 대해서 알려줘")
response

In [ ]:
response.title, response.year, response.director

In [ ]:
class Movies(BaseModel):
    movie:list[Movie] = Field(description="영화 정보의 리스트")

In [ ]:
llm_with_structured = llm.with_structured_output(Movie)
response = llm_with_structured.invoke(
    "2010년대를 대표하는 작품 정보를 알려줘"
)

response

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field

# 1. 중첩된 객체 - 주소 정보
class Address(BaseModel):
    """주소 정보"""
    city: str = Field(description="도시 이름 (예: 서울특별시, 부산광역시)")
    district: str = Field(description="구 또는 군 단위의 명칭 (예: 강남구, 해운대구)")

# 2. 복합 데이터 구조 - 리스트 및 중첩 객체 포함
class PersonDetailed(BaseModel):
    """상세 인적 사항 정보"""
    name: str = Field(description="사람의 전체 이름")
    age: int = Field(description="사람의 만 나이")
    email: Optional[str] = Field(description="이메일 주소 (선택 사항)", default=None)
    address: Address = Field(description="주소 정보를 담고 있는 객체")
    hobbies: List[str] = Field(description="취미 목록 (리스트 형식)")

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

user_input = HumanMessage("""
다음 텍스트에서 사람의 상세 정보를 추출해 주세요.

텍스트: 
peter parker는 28살이며 뉴욕 맨해튼에 살고 있습니다. 
이메일 주소는 james@example.com이고, 
취미로는 독서, 요리, 하이킹을 즐깁니다.
""")

In [ ]:
model_with_structure = llm.with_structured_output(PersonDetailed)
response = model_with_structure.invoke([user_input])
response